In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# MediaPipe setup
mp_face_mesh      = mp.solutions.face_mesh
mp_drawing        = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Project paths
ROOT       = Path("..").resolve()
MODEL_PATH = ROOT / "models" / "face_detector_v2_best.pt"

# Load your face detector
detector = YOLO(str(MODEL_PATH))

# Load MediaPipe Face Mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode        = True,
    max_num_faces            = 5,
    refine_landmarks         = True,
    min_detection_confidence = 0.5,
    min_tracking_confidence  = 0.5
)

print(f"✅ FaceDetector V2 loaded")
print(f"✅ MediaPipe Face Mesh initialized")

In [ ]:
# Key landmark indices per facial feature
LANDMARKS = {
    "left_eye"  : [33, 7, 163, 144, 145, 153, 154, 155,
                   133, 173, 157, 158, 159, 160, 161, 246],
    "right_eye" : [362, 382, 381, 380, 374, 373, 390, 249,
                   263, 466, 388, 387, 386, 385, 384, 398],
    "nose"      : [1, 2, 4, 5, 6, 19, 94, 97, 98,
                   99, 100, 101, 102, 164, 168, 195, 197],
    "lips"      : [61, 84, 17, 314, 405, 320, 307, 375,
                   321, 308, 324, 318, 402, 317, 14, 87,
                   178, 88, 95, 185, 40, 39, 37, 0,
                   267, 269, 270, 409],
    "left_ear"  : [234, 227, 116, 123, 147, 213, 192],
    "right_ear" : [454, 447, 345, 352, 376, 433, 416],
    "jawline"   : [10, 338, 297, 332, 284, 251, 389, 356,
                   454, 323, 361, 288, 397, 365, 379, 378,
                   400, 377, 152, 148, 176, 149, 150, 136,
                   172, 58, 132, 93, 234, 127, 162, 21,
                   54, 103, 67, 109]
}

COLORS = {
    "left_eye"  : (0,   255, 100),
    "right_eye" : (0,   255, 100),
    "nose"      : (255, 200, 0  ),
    "lips"      : (255, 80,  80 ),
    "left_ear"  : (0,   180, 255),
    "right_ear" : (0,   180, 255),
    "jawline"   : (180, 180, 180),
}

print("✅ Landmark map ready")
print(f"   Features tracked: {list(LANDMARKS.keys())}")

In [ ]:
def detect_and_landmark(img_path: str, conf: float = 0.4,
                         padding: float = 0.2):
    """
    Full pipeline:
    1. Load image
    2. FaceDetector V2 → bounding boxes
    3. Crop each face (with padding)
    4. MediaPipe Face Mesh → 468 landmarks per face
    5. Draw everything on original image
    """
    img     = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w    = img.shape[:2]
    output  = img_rgb.copy()

    # Step 1 — Your detector finds faces
    det_results = detector(img, conf=conf, verbose=False)[0]
    boxes       = det_results.boxes

    print(f"FaceDetector V2 → {len(boxes)} face(s) found")

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        det_conf        = float(box.conf[0])

        # Draw detection box
        cv2.rectangle(output, (x1,y1), (x2,y2), (0,255,100), 2)
        cv2.putText(output, f"face {det_conf:.2f}",
                    (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,100), 1)

        # Step 2 — Crop face with padding for better landmark detection
        pad_x = int((x2 - x1) * padding)
        pad_y = int((y2 - y1) * padding)
        cx1   = max(0, x1 - pad_x)
        cy1   = max(0, y1 - pad_y)
        cx2   = min(w, x2 + pad_x)
        cy2   = min(h, y2 + pad_y)

        face_crop     = img_rgb[cy1:cy2, cx1:cx2]
        crop_h, crop_w = face_crop.shape[:2]

        # Step 3 — MediaPipe on crop only
        mesh_results = face_mesh.process(face_crop)

        if not mesh_results.multi_face_landmarks:
            print(f"  Face {i+1}: no landmarks detected")
            continue

        face_lm = mesh_results.multi_face_landmarks[0]
        lm      = face_lm.landmark

        # Step 4 — Draw landmarks mapped back to original image
        for feature, indices in LANDMARKS.items():
            color = COLORS[feature]
            for idx in indices:
                # Convert normalized crop coords → original image coords
                px = int(lm[idx].x * crop_w) + cx1
                py = int(lm[idx].y * crop_h) + cy1
                cv2.circle(output, (px, py), 2, color, -1)

        # Print key points for this face
        nose_x = int(lm[4].x * crop_w) + cx1
        nose_y = int(lm[4].y * crop_h) + cy1
        l_eye_x = int(lm[33].x * crop_w) + cx1
        l_eye_y = int(lm[33].y * crop_h) + cy1
        r_eye_x = int(lm[263].x * crop_w) + cx1
        r_eye_y = int(lm[263].y * crop_h) + cy1
        mouth_x = int(lm[13].x * crop_w) + cx1
        mouth_y = int(lm[13].y * crop_h) + cy1

        print(f"  Face {i+1} landmarks:")
        print(f"    Nose tip  : ({nose_x}, {nose_y})")
        print(f"    Left eye  : ({l_eye_x}, {l_eye_y})")
        print(f"    Right eye : ({r_eye_x}, {r_eye_y})")
        print(f"    Mouth     : ({mouth_x}, {mouth_y})")

    # Show result
    plt.figure(figsize=(12, 8))
    plt.imshow(output)
    plt.title(f"FaceDetector V2 + MediaPipe Landmarks — {len(boxes)} face(s)")
    plt.axis("off")

    # Legend
    legend = [
        plt.Line2D([0],[0], marker='o', color='w',
                   markerfacecolor=tuple(c/255 for c in col),
                   markersize=8, label=feat)
        for feat, col in COLORS.items()
    ]
    plt.legend(handles=legend, loc='lower right', fontsize=9)
    plt.tight_layout()

    save_path = ROOT / "results" / "landmarks_output.png"
    plt.savefig(str(save_path), dpi=120)
    plt.show()
    print(f"\nSaved → {save_path}")

print("✅ Pipeline function ready")

In [ ]:
detect_and_landmark(r"C:\Users\Sag\Downloads\ChatGPT Image May 25, 2026, 08_19_41 PM.png")

In [ ]:
# Re-initialize face mesh for multiple faces
face_mesh_multi = mp_face_mesh.FaceMesh(
    static_image_mode        = True,
    max_num_faces            = 10,
    refine_landmarks         = True,
    min_detection_confidence = 0.4,
    min_tracking_confidence  = 0.4
)

detect_and_landmark(r"C:\Users\Sag\Downloads\WhatsApp Image 2026-06-06 at 2.41.08 PM.jpeg")

In [ ]:
face_mesh_video = mp_face_mesh.FaceMesh(
    static_image_mode        = False,
    max_num_faces            = 3,
    refine_landmarks         = True,
    min_detection_confidence = 0.5,
    min_tracking_confidence  = 0.5
)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Webcam started — press Q to quit")

frame_count  = 0
fps_start    = cv2.getTickCount()
cached_boxes = []          # reuse boxes between YOLO runs
DETECT_EVERY = 3           # run YOLO every 3rd frame only

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w       = frame.shape[:2]
    frame_count += 1

    # Run YOLO only every Nth frame
    if frame_count % DETECT_EVERY == 0:
        small       = cv2.resize(frame, (320, 240))
        det_results = detector(small, conf=0.4, verbose=False)[0]
        # Scale boxes back to full resolution
        cached_boxes = []
        for box in det_results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cached_boxes.append([
                int(x1 * 2), int(y1 * 2),
                int(x2 * 2), int(y2 * 2),
                float(box.conf[0])
            ])

    # Draw cached boxes + run landmarks every frame
    for (x1, y1, x2, y2, conf_score) in cached_boxes:
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,100), 2)

        # Crop with padding
        pad_x = int((x2-x1) * 0.2)
        pad_y = int((y2-y1) * 0.2)
        cx1   = max(0, x1-pad_x)
        cy1   = max(0, y1-pad_y)
        cx2   = min(w, x2+pad_x)
        cy2   = min(h, y2+pad_y)

        face_crop      = img_rgb[cy1:cy2, cx1:cx2]
        crop_h, crop_w = face_crop.shape[:2]

        if crop_w < 10 or crop_h < 10:
            continue

        mesh_result = face_mesh_video.process(face_crop)
        if not mesh_result.multi_face_landmarks:
            continue

        lm = mesh_result.multi_face_landmarks[0].landmark

        for feature, indices in LANDMARKS.items():
            color = COLORS[feature]
            for idx in indices:
                px = int(lm[idx].x * crop_w) + cx1
                py = int(lm[idx].y * crop_h) + cy1
                cv2.circle(frame, (px, py), 2, color, -1)

    # FPS counter
    elapsed = (cv2.getTickCount() - fps_start) / cv2.getTickFrequency()
    fps     = frame_count / elapsed
    cv2.putText(frame, f"FPS:{fps:.1f} Faces:{len(cached_boxes)}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 200, 255), 2)

    cv2.imshow("FaceDetector V2 + Landmarks", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Average FPS: {fps:.1f}")